# Jacobian lens on GPT-2

The same pipeline as the logit lens notebooks (`logit_lens_examples.ipynb`, `logit_lens_dataset.ipynb`),
but every block output is first transported into the final-layer basis:

```
lens_l(h) = unembed( J_l @ h ),   J_l = E[∂h_final / ∂h_l]
```

`J_l` is the average input–output Jacobian over a mixed corpus (`jlens.fit`); with `J_l = I` this is exactly the logit lens.
The notebook fits the lens, checks it, shows examples, and runs both lenses over `data/evaluations` side by side.

## Setup

In [ ]:
import os
import sys

REPO_URL = "https://github.com/ai360-project-school-j-lens/jacobian-lens-gpt2.git"
REPO_DIR = os.path.abspath("jacobian-lens-gpt2")

if os.path.isfile("gpt2/lens_eval.py"):  # ноутбук открыт прямо из репозитория
    REPO_DIR = os.getcwd()
elif os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}
sys.path.append(REPO_DIR)

In [ ]:
from functools import partial

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

import jlens
from gpt2 import GPT2LensModel
from gpt2.lens_eval import (
    DATASETS,
    FIT_MIX,
    ROLE_NAMES,
    evaluate_all,
    first_layer,
    identity_lens,
    load_fit_prompts,
    jacobian_lens,
    layer_hit_rate,
    layer_mean,
    layer_median_rank,
    load_eval,
    logit_lens,
    pass_at_k,
    plot_layer_curves,
    readout_position,
    show_lens,
    top_tokens_table,
)

jlens.configure_logging()
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

gpt = GPT2LensModel.from_pretrained(device=DEVICE)
logit = partial(logit_lens, gpt)
gpt

In [ ]:
evals = {dataset: load_eval(REPO_DIR, dataset) for dataset in DATASETS}

K = 10  # главный порог ранга для hit / pass@k
LAST = gpt.n_layers - 1  # последний слой — выход модели

## 1. Fit (or load) the lens

### 1a. Fit corpus

`J_l` is an average over the corpus, so the lens only reads well on text that looks like what it was fitted on.
Instead of WikiText alone (encyclopedic English), the corpus mixes genres close to the evaluation sets:
web text (FineWeb), Wikipedia (WikiText), fiction (TinyStories), dialogue (UltraChat), math word problems (GSM8K),
code (HumanEval), poetry, and Spanish / French / German / Portuguese Wikipedia (for multilingual).
The evaluation prompts themselves are **not** in the corpus. Change `FIT_MIX` to rebalance; a source that fails
to load is skipped with a message.

In [ ]:
FIT_MIX

In [ ]:
fit_prompts = load_fit_prompts(FIT_MIX)
fit_prompts.source.value_counts()

In [ ]:
fit_prompts.groupby("source").head(1).assign(text=lambda f: f.text.str[:300])

### 1b. Fit

`fit(model, prompts)` computes `J_l` for layers 0..10 (target = the last block); every prompt is truncated to
`max_seq_len` tokens. `dim_batch` is the memory knob — each prompt does `ceil(768 / dim_batch)` backward passes.
The fit is checkpointed and resumes after a restart (the prompt list must stay the same); the result is saved
to `LENS_PATH` and loaded on the next run.

In [ ]:
LENS_PATH = f"gpt2_jacobian_lens_mix{len(fit_prompts)}.pt"

if os.path.exists(LENS_PATH):
    lens = jlens.JacobianLens.load(LENS_PATH)
else:
    lens = jlens.fit(
        gpt,
        fit_prompts.text.tolist(),
        dim_batch=32,
        max_seq_len=128,
        checkpoint_path=LENS_PATH.replace(".pt", "_ckpt.pt"),
    )
    lens.save(LENS_PATH)

jacobian = partial(jacobian_lens, gpt, lens)
lenses = {"logit lens": logit, "J-lens": jacobian}
lens

## 2. Sanity checks

1. `jacobian_lens` with `J_l = I` must equal the handwritten logit lens — the hooks on `gpt.layers` (used by `fit`)
   see exactly the block outputs that `hidden_states` returns, so the two lenses are aligned layer by layer.
2. The last row of the J-lens is the model output (≈ 0 difference with the logit lens there).
3. `jacobian_lens` agrees with the library's `lens.apply`.

In [ ]:
prompt = evals["order-ops"][0]["prompt"].rstrip()

(jacobian_lens(gpt, identity_lens(gpt), prompt) - logit(prompt)).abs().max().item()

In [ ]:
(jacobian(prompt)[-1] - logit(prompt)[-1]).abs().max().item()

In [ ]:
lens_logits, _, _ = lens.apply(gpt, prompt)
mine = jacobian(prompt).cpu()
max((mine[layer] - lens_logits[layer]).abs().max().item() for layer in lens.source_layers)

How far each `J_l` is from the identity, i.e. from the logit lens:

In [ ]:
eye = torch.eye(gpt.d_model)
distance = pd.Series(
    {layer: ((J - eye).norm() / eye.norm()).item() for layer, J in lens.jacobians.items()}, name="‖J − I‖ / ‖I‖"
)
distance.plot(marker="o", title="Distance of J_l from the identity", xlabel="layer", ylabel="‖J − I‖ / ‖I‖", grid=True)
plt.show()

## 3. Visualisation

`show_lens` is the same position × layer view as for the logit lens, only with the J-lens readout.

### Dataset prompts

In [ ]:
for sample in evals["order-ops"][0:5]:
    show_lens(
        gpt,
        jacobian,
        sample["prompt"].rstrip(),
        title=f"J-lens: {sample['name']}",
        description=f"target: {sample.get('target')!r}, intermediates: {sample['intermediates']}",
    )

### One prompt from every dataset

In [ ]:
for dataset, items in evals.items():
    sample = items[0]
    show_lens(
        gpt,
        jacobian,
        sample["prompt"].rstrip(),
        title=f"J-lens — {dataset}: {sample['name']}",
        description=f"target: {sample.get('target')!r}, intermediates: {sample['intermediates']}",
    )

### Top-5 at the readout position: logit lens vs J-lens

In [ ]:
for dataset in ["multihop", "multilingual", "poetry", "association"]:
    sample = evals[dataset][0]
    prompt = sample["prompt"].rstrip()
    position = readout_position(gpt.tokenizer, gpt.encode(prompt)[0].tolist(), dataset)
    print(f"{dataset}: {prompt!r}\nintermediates: {sample['intermediates']}, target: {sample.get('target')!r}")
    display(top_tokens_table(gpt, lenses, prompt, position=position))

### Custom prompt

In [ ]:
custom_prompt = "The Eiffel Tower is located in the city of"
show_lens(gpt, logit, custom_prompt, title="Logit lens")
show_lens(gpt, jacobian, custom_prompt, title="J-lens")
top_tokens_table(gpt, lenses, custom_prompt)

## 4. Dataset evaluation: J-lens vs logit lens

**Protocol** (`data/evaluations/README.md`):

* **readout position** — one token per prompt: the last prompt token (the one right before `target`,
  the closing period, the tail of the typo); for poetry — the newline at the end of the first line;
* **rank** — 1-based rank of a word among all 50 257 tokens at that position, min over its single-token
  spellings (with/without leading space, case); for order-ops — min over the synonym set
  (numbers → digits and words, operations → symbols and words); multi-token words fall back to their first token;
* **pass@k** — mean over items of the fraction of `intermediates` whose min-over-layers rank ≤ k;
* **control** — the same ranks for the intermediates of *another* item of the same dataset.
  It is the chance level: a lens that surfaces any vaguely topical word scores here too.

The last row of both lenses is the same model output, so the comparison that matters is over the inner layers 0..10
(`inner_best` below).

In [ ]:
results = {name: evaluate_all(gpt, lens_fn, evals) for name, lens_fn in lenses.items()}
words = pd.concat([w.assign(lens=name) for name, (w, _) in results.items()], ignore_index=True)
items = pd.concat([i.assign(lens=name) for name, (_, i) in results.items()], ignore_index=True)
words["inner_best"] = [int(r[:-1].min()) for r in words.ranks]
inter = words[words.kind == "intermediate"]

### 4a. pass@k

In [ ]:
scores = pass_at_k(words, layers=slice(None, -1))
scores[scores.kind == "intermediate"].pivot_table(index="dataset", columns=["k", "lens"], values="score").round(3)

In [ ]:
at_k = scores[scores.k == K].pivot_table(index="dataset", columns=["kind", "lens"], values="score")
at_k.round(3)

In [ ]:
at_k[[("intermediate", "logit lens"), ("intermediate", "J-lens"), ("control", "logit lens"), ("control", "J-lens")]].plot.bar(
    figsize=(11, 4), rot=0, title=f"pass@{K} over layers 0..{LAST - 1}"
)
plt.ylabel(f"pass@{K}")
plt.show()

### 4b. At what depth

In [ ]:
def curves(stat, kinds=("intermediate", "control")):
    return {
        f"{name}: {kind}": stat(words[(words.lens == name) & (words.kind == kind)])
        for name in lenses
        for kind in kinds
    }


plot_layer_curves(
    curves(lambda w: layer_hit_rate(w, K)),
    title=f"Share of words with rank ≤ {K} at the readout position",
    ylabel=f"rank ≤ {K}",
)
plot_layer_curves(curves(layer_median_rank), title="Median rank at the readout position", ylabel="median rank", logy=True)
plot_layer_curves(
    curves(lambda w: layer_hit_rate(w, K), kinds=("target",)),
    title=f"Target at rank ≤ {K} (multihop, multilingual, order-ops)",
    ylabel=f"rank ≤ {K}",
)

### 4c. General behaviour over all prompt positions

In [ ]:
behaviour = {
    name: {column: layer_mean(items[items.lens == name], column) for column in ["agreement", "copy_rate", "kl_to_final"]}
    for name in lenses
}
plot_layer_curves(
    {f"{name}: top-1 == model top-1": behaviour[name]["agreement"] for name in lenses}
    | {f"{name}: top-1 == input token": behaviour[name]["copy_rate"] for name in lenses},
    title="Over all prompt positions",
    ylabel="share of positions",
)
plot_layer_curves(
    {f"{name}: KL(model || layer)": behaviour[name]["kl_to_final"] for name in lenses},
    title="KL to the model output",
    ylabel="nats",
)

### 4d. Head to head: best inner-layer rank of every intermediate

In [ ]:
h2h = inter.pivot_table(index=["dataset", "item", "word"], columns="lens", values="inner_best")
h2h["winner"] = np.select(
    [h2h["J-lens"] < h2h["logit lens"], h2h["J-lens"] > h2h["logit lens"]], ["J-lens", "logit lens"], "tie"
)
# во сколько раз (в порядках величины) J-lens поднимает слово выше
h2h["log10 gain"] = np.log10(h2h["logit lens"]) - np.log10(h2h["J-lens"])

summary = h2h.groupby("dataset").winner.value_counts().unstack(fill_value=0)
summary["median log10 gain"] = h2h.groupby("dataset")["log10 gain"].median()
summary.round(2)

In [ ]:
h2h.sort_values("log10 gain", ascending=False).head(15)

In [ ]:
h2h.sort_values("log10 gain").head(10)

### 4e. By role (multilingual, order-ops)

In [ ]:
roles = inter[inter.dataset.isin(list(ROLE_NAMES))].assign(hit=lambda w: w.inner_best <= K)
roles["role_name"] = [ROLE_NAMES[d][r] for d, r in zip(roles.dataset, roles.role)]
roles.pivot_table(index=["dataset", "role_name"], columns="lens", values="hit").round(3)

### 4f. In the prompt or not

In [ ]:
inter.assign(hit=inter.inner_best <= K).pivot_table(index="dataset", columns=["in_prompt", "lens"], values="hit").round(3)

### 4g. What each lens reads at the readout position

In [ ]:
readout = items.set_index(["dataset", "item", "lens"]).readout_top1.apply(pd.Series)
readout.columns.name = "layer"
first_items = items.groupby("dataset").head(1).item
readout[readout.index.get_level_values("item").isin(first_items)]

## 5. Conclusions

Generated from the numbers above, so they stay correct after a re-run (other `FIT_MIX`, `K`, model size).

In [ ]:
def pct(x):
    return "n/a" if x is None or pd.isna(x) else f"{100 * x:.0f}%"


def layer(x):
    return "never" if x is None else f"L{x}"

In [ ]:
lines = []
for dataset in at_k.index:
    row = at_k.loc[dataset]
    j, l = row[("intermediate", "J-lens")], row[("intermediate", "logit lens")]
    jc, lc = row.get(("control", "J-lens")), row.get(("control", "logit lens"))
    verdict = "J-lens better" if j > l + 0.05 else "logit lens better" if l > j + 0.05 else "about the same"
    lines.append(
        f"{dataset}: inner-layer pass@{K} J-lens {pct(j)} (control {pct(jc)}) vs logit lens {pct(l)} (control {pct(lc)}) — {verdict}; "
        f"J-lens margin over its control {100 * (j - jc):+.0f} pp, logit lens {100 * (l - lc):+.0f} pp."
    )

for dataset, row in summary.iterrows():
    lines.append(
        f"{dataset}: per word, J-lens ranks the intermediate higher in {row.get('J-lens', 0)} cases, lower in "
        f"{row.get('logit lens', 0)}, ties {row.get('tie', 0)}; median gain {row['median log10 gain']:+.2f} orders of magnitude."
    )

inter_curves = {name: layer_hit_rate(inter[inter.lens == name], K) for name in lenses}
for dataset in inter_curves["J-lens"].index:
    peaks = {name: curve.loc[dataset].iloc[:-1] for name, curve in inter_curves.items()}
    lines.append(
        f"{dataset}: intermediates peak at " + ", ".join(
            f"{name} L{peak.idxmax()} ({pct(peak.max())})" for name, peak in peaks.items()
        ) + "."
    )

for name in lenses:
    agreement, copy_rate, kl = (behaviour[name][c] for c in ["agreement", "copy_rate", "kl_to_final"])
    lines.append(
        f"{name}, averaged over datasets: top-1 == model top-1 on ≥50% of positions from "
        f"{layer(first_layer(agreement.mean(), 0.5))}, KL ≤ 1 nat from {layer(first_layer(kl.mean(), 1.0, above=False))}, "
        f"copy rate at L0 {pct(copy_rate.mean()[0])}."
    )

hit_by_prompt = inter.assign(hit=inter.inner_best <= K).groupby(["lens", "in_prompt"]).hit.mean()
for name in lenses:
    lines.append(
        f"{name}: hit rate for intermediates in the prompt {pct(hit_by_prompt.get((name, True)))}, "
        f"absent from it {pct(hit_by_prompt.get((name, False)))}."
    )

for dataset, group in roles.groupby("dataset"):
    by_role = group.pivot_table(index="role_name", columns="lens", values="hit")
    lines.append(
        f"{dataset} by role (J-lens / logit lens): "
        + ", ".join(f"{role} {pct(r['J-lens'])} / {pct(r['logit lens'])}" for role, r in by_role.iterrows()) + "."
    )

for line in lines:
    print("•", line)

### How to read these numbers

* **J-lens vs logit lens** is a fair fight only over layers 0..10 — the output row is identical. A J-lens win means the transport
  `J_l` turns a mid-layer residual into something the unembedding can read, where the logit lens sees noise.
* **Margin over control** matters more than the raw pass@k: a lens that makes every layer look like the output will raise
  the control as well.
* **Agreement / KL / copy rate.** Expect the J-lens to agree with the output from earlier layers and to copy the input token less —
  that is what the transport is for. If it does not, check `‖J − I‖` (§2): a lens close to the identity is just the logit lens.
* **Fit corpus.** The lens is only as good as the corpus `J_l` is averaged over: re-run with a different `FIT_MIX`
  (more prompts, or a genre removed) to see which part of the corpus a dataset depends on.
* **GPT-2 small.** If the model fails the task (accuracy in `logit_lens_dataset.ipynb`, §2), there may be no intermediate to find —
  neither lens can show a step the model does not compute.